# 04 — Evaluation V2

En este notebook evaluaremos en profundidad el mejor modelo entrenado (`best_model.joblib`). Al igual que en la versión 1, analizaremos el equilibrio entre Precision y Recall, evaluaremos distintos umbrales de decisión, analizaremos los errores (Falsos Positivos y Falsos Negativos), y utilizaremos SHAP para la interpretabilidad del modelo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, average_precision_score)
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Carga de datos y modelo

In [ ]:
# Cargamos el modelo
best_model = joblib.load("../models/best_model.joblib")
print(f"Modelo cargado: {type(best_model).__name__}")

# Cargamos los datos
X_test = pd.read_csv("../data/processed/X_test_v2.csv")
y_test = pd.read_csv("../data/processed/y_test_v2.csv")["target"]
X_train = pd.read_csv("../data/processed/X_train_v2.csv")
print(f"X_test shape: {X_test.shape}")

## 2. Curva Precision-Recall

La curva Precision-Recall es ideal para datasets con desbalance de clases o cuando, como en nuestro caso, **nos importan mucho los verdaderos positivos (Hot Leads)** y queremos controlar los falsos positivos.

- **Precision:** De todos los leads que el modelo dice que son Hot, ¿cuántos lo son realmente?
- **Recall (Sensibilidad):** De todos los Hot Leads reales, ¿cuántos logra atrapar el modelo?

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color="#8e44ad", lw=2, label=f"PR Curve (AP = {ap:.4f})")
ax.set_title("Curva Precision-Recall (v2)")
ax.set_xlabel("Recall (Proporción de Hot Leads atrapados)")
ax.set_ylabel("Precision (Precisión de los leads marcados como Hot)")
ax.axhline(y_test.mean(), color="gray", linestyle="--", label="Random Baseline")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Análisis de umbral de decisión

Por defecto, los modelos usan un umbral del 50% (0.50). Sin embargo, el equipo de ventas prefiere no perder *Hot Leads*, aunque eso implique llamar a algunos *Cold Leads* por error. Por lo tanto, analizamos el efecto de bajar el umbral.

In [ ]:
thresholds_to_test = [0.30, 0.35, 0.40, 0.50, 0.60]

print("=== IMPACTO DEL UMBRAL DE DECISIÓN ===\n")
print(f"{'Umbral':^8s} | {'Precision':^10s} | {'Recall':^8s} | {'F1-Score':^8s}")
print("-" * 45)

for thresh in thresholds_to_test:
    y_pred_th = (y_proba >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_th)
    rec = recall_score(y_test, y_pred_th)
    f1 = f1_score(y_test, y_pred_th)
    print(f"{thresh:>8.2f} | {prec:>10.4f} | {rec:>8.4f} | {f1:>8.4f}")

# Fijamos el umbral de negocio en 0.35
UMBRAL = 0.35
y_pred = (y_proba >= UMBRAL).astype(int)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cold (0)", "Hot (1)"])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Matriz de Confusión (Umbral {UMBRAL})")
plt.grid(False)
plt.show()

## 4. Análisis de errores

Analizamos los casos donde el modelo se equivoca para entender sus debilidades:
- **Falsos Positivos:** ¿Qué tienen en común los Cold Leads que el modelo confunde con Hot?
- **Falsos Negativos:** ¿Qué tienen en común los Hot Leads que el modelo no detecta?

In [ ]:
test_analysis = X_test.copy()
test_analysis["y_real"] = y_test.values
test_analysis["y_pred"] = y_pred
test_analysis["y_proba"] = y_proba

test_analysis["tipo_resultado"] = "TN"
test_analysis.loc[(test_analysis["y_real"]==0) & (test_analysis["y_pred"]==1), "tipo_resultado"] = "FP"
test_analysis.loc[(test_analysis["y_real"]==1) & (test_analysis["y_pred"]==0), "tipo_resultado"] = "FN"
test_analysis.loc[(test_analysis["y_real"]==1) & (test_analysis["y_pred"]==1), "tipo_resultado"] = "TP"

print("Distribución de resultados:")
print(test_analysis["tipo_resultado"].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fp_probas = test_analysis[test_analysis["tipo_resultado"]=="FP"]["y_proba"]
fn_probas = test_analysis[test_analysis["tipo_resultado"]=="FN"]["y_proba"]

axes[0].hist(fp_probas, bins=20, color="#e74c3c", edgecolor="black", alpha=0.7)
axes[0].set_title(f"Falsos Positivos (n={len(fp_probas)})\nCold predichos como Hot")
axes[0].set_xlabel("Probabilidad predicha")
axes[0].set_ylabel("Frecuencia")
axes[0].axvline(x=0.5, color="gray", linestyle="--")

axes[1].hist(fn_probas, bins=20, color="#f39c12", edgecolor="black", alpha=0.7)
axes[1].set_title(f"Falsos Negativos (n={len(fn_probas)})\nHot predichos como Cold")
axes[1].set_xlabel("Probabilidad predicha")
axes[1].set_ylabel("Frecuencia")
axes[1].axvline(x=0.5, color="gray", linestyle="--")

plt.tight_layout()
plt.show()

print(f"Falsos Positivos: probabilidad media = {fp_probas.mean():.3f}")
print(f"Falsos Negativos: probabilidad media = {fn_probas.mean():.3f}")

## 5. Resumen de métricas finales

In [ ]:
print("=" * 60)
print("       RESUMEN FINAL DEL MODELO")
print("=" * 60)
print(f"\nModelo: {type(best_model).__name__}")
print(f"Features: {X_test.shape[1]}")
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

print(f"\n--- Métricas en Test (umbral = {UMBRAL}) ---")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

fp = len(fp_probas)
fn = len(fn_probas)
print(f"\n--- Errores ---")
print(f"  Falsos Positivos: {fp} ({fp/len(y_test)*100:.1f}%) → Cold que el modelo dice Hot")
print(f"  Falsos Negativos: {fn} ({fn/len(y_test)*100:.1f}%) → Hot que el modelo pierde")

## 6. Interpretabilidad con SHAP

Sabemos **qué features usa más el modelo**, pero ahora con SHAP analizamos **cómo** las usa ni **en qué dirección** afectan cada predicción.

SHAP calcula exactamente cuánto aporta cada feature a empujar la probabilidad hacia *Hot* o hacia *Cold*.

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_test)
print(f"SHAP values calculados para {X_test.shape[0]:,} leads x {X_test.shape[1]} features")

### 6.1 Importancia global (Bar plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(shap_values[:, :, 1], max_display=15, show=False)
plt.title("SHAP — Top features por impacto promedio en la predicción")
plt.tight_layout()
plt.show()

### 6.2 Summary plot (gráfico de abejas)

Cada punto es un lead real del test set. Funciona así:
- **Derecha** → esa feature empuja hacia **Hot**
- **Izquierda** → esa feature empuja hacia **Cold**
- **Color rojo** = valor alto de la feature, **azul** = valor bajo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values[:, :, 1], X_test, max_display=15, show=False)
plt.title("SHAP Summary Plot — Efecto de cada feature en la predicción")
plt.tight_layout()
plt.show()

### 6.3 Análisis de un lead individual (Waterfall plot)

Analizamos cómo el modelo tomó la decisión para el primer lead del dataset de test.

In [ ]:
# Tomamos el índice 0 del test set
lead_idx = 0
lead_real = y_test.iloc[lead_idx]
lead_pred_prob = y_proba[lead_idx]

print(f"Lead seleccionado: Index {lead_idx}")
print(f"Predicción del modelo: {lead_pred_prob:.1%} de ser Hot")
print(f"Resultado real: {'Hot (1)' if lead_real == 1 else 'Cold (0)'}")

shap.plots.waterfall(shap_values[lead_idx, :, 1], max_display=10)